# ShanghaiTech YOLO + CLIP + kNN (Frame-Based VAD)

This notebook builds a simple object-centric anomaly detection baseline for the ShanghaiTech Campus dataset:

1. Download the Kaggle train and test splits.
2. Normalize the dataset layout under `/content/data/shanghaitech`.
3. Extract frames when needed.
4. Detect objects with YOLO.
5. Encode object crops with CLIP.
6. Score test frames with a FAISS kNN index.

This version does not use temporal smoothing, tracking, or optical flow.


In [ ]:
%%capture
%pip install -q ultralytics open_clip_torch faiss-cpu opencv-python-headless scikit-learn kaggle Pillow tqdm


In [ ]:
from pathlib import Path

DATA_ROOT = Path("/content/data/shanghaitech")
DOWNLOAD_DIR = Path("/content/datasets")
TMP_DIR = Path("/content/tmp_shanghaitech")
OUTPUT_ROOT = Path("/content/artifacts/clip_knn_baseline/shanghaitech")

DETECTOR_WEIGHTS = "yolov8n.pt"
CLIP_MODEL = "ViT-B-32"
CLIP_PRETRAINED = "laion2b_s34b_b79k"
KEEP_CLASSES = ["person", "bicycle", "car", "motorcycle", "bus", "truck", "skateboard"]
DETECTION_CONF = 0.25
MIN_BOX_AREA = 16 * 16
PADDING_RATIO = 0.05
CLIP_BATCH_SIZE = 32
IMG_SIZE = 640
K_NEIGHBORS = 5
MAX_TRAIN_FEATURES = 200_000
VIDEO_LIMIT = 0
FRAME_LIMIT = 0
USE_FULL_FRAME_FALLBACK = True

for path in (DATA_ROOT, DOWNLOAD_DIR, TMP_DIR, OUTPUT_ROOT):
    path.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)


## Kaggle authentication

If you already uploaded `kaggle.json` in this runtime, the next cell will reuse it.


In [ ]:
import os
import stat
from google.colab import files

KAGGLE_DIR = Path.home() / ".kaggle"
KAGGLE_DIR.mkdir(parents=True, exist_ok=True)
kaggle_json_path = KAGGLE_DIR / "kaggle.json"

if not kaggle_json_path.exists():
    uploaded = files.upload()
    if "kaggle.json" not in uploaded:
        raise RuntimeError("Please upload a file named kaggle.json")
    kaggle_json_path.write_bytes(uploaded["kaggle.json"])

os.chmod(kaggle_json_path, stat.S_IRUSR | stat.S_IWUSR)
print("Kaggle credentials ready at", kaggle_json_path)


In [ ]:
!kaggle datasets download -d nikanvasei/shanghaitech-campus-dataset -p {DOWNLOAD_DIR}
!kaggle datasets download -d nikanvasei/shanghaitech-campus-dataset-test -p {DOWNLOAD_DIR}
!ls -lh {DOWNLOAD_DIR}


In [ ]:
import shutil
import zipfile

import cv2
import numpy as np


def unzip_archives(archive_paths, tmp_root):
    extracted_roots = []
    for archive_path in archive_paths:
        target_dir = tmp_root / archive_path.stem
        target_dir.mkdir(parents=True, exist_ok=True)
        if not any(target_dir.iterdir()):
            with zipfile.ZipFile(archive_path, "r") as handle:
                handle.extractall(target_dir)
        extracted_roots.append(target_dir)
    return extracted_roots


def copy_tree(src: Path, dst: Path):
    dst.mkdir(parents=True, exist_ok=True)
    for item in src.iterdir():
        target = dst / item.name
        if item.is_dir():
            shutil.copytree(item, target, dirs_exist_ok=True)
        else:
            shutil.copy2(item, target)


def find_dirs_named(root: Path, name: str):
    return sorted(
        [path for path in root.rglob(name) if path.is_dir()],
        key=lambda path: (len(path.parts), str(path)),
    )


def normalize_dataset_layout(extracted_roots, dataset_root: Path):
    for root in extracted_roots:
        for split_name in ("training", "testing"):
            for match in find_dirs_named(root, split_name):
                copy_tree(match, dataset_root / split_name)
        for match in find_dirs_named(root, "test_frame_mask"):
            copy_tree(match, dataset_root / "testing" / "test_frame_mask")


def frame_sort_key(path: Path):
    stem = path.stem
    return (int(stem), stem) if stem.isdigit() else (10**9, stem)


def iter_frame_files(video_dir: Path):
    for pattern in ("*.jpg", "*.jpeg", "*.png", "*.bmp"):
        yield from video_dir.glob(pattern)


def has_existing_frames(frames_root: Path):
    if not frames_root.exists():
        return False
    return any(frames_root.glob("*/*.jpg")) or any(frames_root.glob("*/*.png"))


def extract_frames_from_video(video_path: Path, output_dir: Path, overwrite: bool = False):
    output_dir.mkdir(parents=True, exist_ok=True)
    existing_frames = sorted(output_dir.glob("*.jpg"), key=frame_sort_key)
    if existing_frames and not overwrite:
        return len(existing_frames)

    if overwrite:
        for frame_path in existing_frames:
            frame_path.unlink()

    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    frame_count = 0
    success, frame = capture.read()
    while success:
        cv2.imwrite(str(output_dir / f"{frame_count}.jpg"), frame)
        frame_count += 1
        success, frame = capture.read()

    capture.release()
    return frame_count


def ensure_frame_directories(dataset_root: Path, overwrite: bool = False):
    for split_name in ("training", "testing"):
        split_root = dataset_root / split_name
        frames_root = split_root / "frames"
        videos_root = split_root / "videos"

        if has_existing_frames(frames_root):
            continue
        if not videos_root.exists():
            continue

        frames_root.mkdir(parents=True, exist_ok=True)
        video_paths = sorted([path for path in videos_root.iterdir() if path.is_file()], key=lambda path: path.name)
        for video_path in video_paths:
            extract_frames_from_video(video_path, frames_root / video_path.stem, overwrite=overwrite)


def save_clip_lengths(dataset_root: Path):
    split_map = {"training": "train_clip_lengths.npy", "testing": "test_clip_lengths.npy"}
    for split_name, output_name in split_map.items():
        frames_root = dataset_root / split_name / "frames"
        if not frames_root.exists():
            continue
        total = 0
        clip_lengths = []
        for video_dir in sorted([path for path in frames_root.iterdir() if path.is_dir()], key=lambda path: path.name):
            frame_paths = sorted(iter_frame_files(video_dir), key=frame_sort_key)
            total += len(frame_paths)
            clip_lengths.append(total)
        np.save(dataset_root / output_name, np.asarray(clip_lengths, dtype=np.int64))


def print_dataset_summary(dataset_root: Path):
    for split_name in ("training", "testing"):
        frames_root = dataset_root / split_name / "frames"
        videos = sorted([path for path in frames_root.iterdir() if path.is_dir()], key=lambda path: path.name) if frames_root.exists() else []
        frame_count = 0
        for video_dir in videos:
            frame_count += len(list(iter_frame_files(video_dir)))
        print(f"{split_name}: {len(videos)} videos, {frame_count} frames")

    mask_root = dataset_root / "testing" / "test_frame_mask"
    if mask_root.exists():
        print("test_frame_mask files:", len(list(mask_root.glob("*.npy"))))
    else:
        print("test_frame_mask files: 0")


In [ ]:
archive_paths = sorted(DOWNLOAD_DIR.glob("*.zip"))
if not archive_paths:
    raise RuntimeError(f"No zip archives found in {DOWNLOAD_DIR}")

extracted_roots = unzip_archives(archive_paths, TMP_DIR)
normalize_dataset_layout(extracted_roots, DATA_ROOT)
ensure_frame_directories(DATA_ROOT)
save_clip_lengths(DATA_ROOT)
print_dataset_summary(DATA_ROOT)


In [ ]:
import csv
import json
import sys
from dataclasses import dataclass
from typing import List, Sequence

import faiss
import open_clip
import torch
from PIL import Image
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm
from ultralytics import YOLO


@dataclass(frozen=True)
class FrameRecord:
    video_name: str
    frame_index: int
    frame_path: Path


def collect_frame_records(dataset_root: Path, split: str):
    split_dir = "training" if split == "train" else "testing"
    frames_root = dataset_root / split_dir / "frames"
    frame_records = []
    for video_dir in sorted([path for path in frames_root.iterdir() if path.is_dir()], key=lambda path: path.name):
        frame_paths = sorted(iter_frame_files(video_dir), key=frame_sort_key)
        for frame_index, frame_path in enumerate(frame_paths):
            frame_records.append(FrameRecord(video_name=video_dir.name, frame_index=frame_index, frame_path=frame_path))
    return frame_records


def limit_records(records, video_limit: int, frame_limit: int):
    if video_limit > 0:
        keep_videos = []
        for record in records:
            if record.video_name not in keep_videos:
                keep_videos.append(record.video_name)
            if len(keep_videos) == video_limit:
                break
        keep_videos = set(keep_videos)
        records = [record for record in records if record.video_name in keep_videos]
    if frame_limit > 0:
        records = list(records[:frame_limit])
    return list(records)


def resolve_class_ids(detector: YOLO, class_names: Sequence[str]):
    detector_names = detector.model.names
    normalized = {str(name).strip().lower(): idx for idx, name in detector_names.items()}
    class_ids = []
    missing = []
    for class_name in class_names:
        key = class_name.strip().lower()
        if key in normalized:
            class_ids.append(normalized[key])
        else:
            missing.append(class_name)
    if missing:
        raise ValueError(f"Unknown detector classes: {', '.join(missing)}")
    return class_ids


def crop_box(image_bgr: np.ndarray, box: np.ndarray, padding_ratio: float):
    height, width = image_bgr.shape[:2]
    x1, y1, x2, y2 = box.astype(np.float32)
    pad_x = (x2 - x1) * padding_ratio
    pad_y = (y2 - y1) * padding_ratio
    x1 = max(0, int(np.floor(x1 - pad_x)))
    y1 = max(0, int(np.floor(y1 - pad_y)))
    x2 = min(width, int(np.ceil(x2 + pad_x)))
    y2 = min(height, int(np.ceil(y2 + pad_y)))
    crop = image_bgr[y1:y2, x1:x2]
    if crop.size == 0:
        crop = image_bgr
    return cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)


def clip_encode(images, clip_model, preprocess, device: str, batch_size: int):
    encoded_batches = []
    for start in range(0, len(images), batch_size):
        batch_images = images[start:start + batch_size]
        tensors = [preprocess(Image.fromarray(image_rgb)) for image_rgb in batch_images]
        batch_tensor = torch.stack(tensors, dim=0).to(device)
        with torch.no_grad():
            features = clip_model.encode_image(batch_tensor)
            features = features / features.norm(dim=-1, keepdim=True)
        encoded_batches.append(features.detach().cpu().numpy().astype(np.float32))
    return np.concatenate(encoded_batches, axis=0)


def save_metadata(output_path: Path, metadata_rows):
    with output_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(["frame_id", "video_name", "frame_index", "frame_path", "num_regions"])
        writer.writerows(metadata_rows)


def extract_split_features(split: str, overwrite: bool = False):
    split_output_dir = OUTPUT_ROOT / split
    split_output_dir.mkdir(parents=True, exist_ok=True)

    features_path = split_output_dir / "features.npy"
    boxes_path = split_output_dir / "boxes.npy"
    classes_path = split_output_dir / "classes.npy"
    metadata_path = split_output_dir / "metadata.csv"

    if features_path.exists() and boxes_path.exists() and classes_path.exists() and metadata_path.exists() and not overwrite:
        print(f"Skipping {split}: cached features already exist at {split_output_dir}")
        return

    frame_records = collect_frame_records(DATA_ROOT, split)
    frame_records = limit_records(frame_records, VIDEO_LIMIT, FRAME_LIMIT)
    class_ids = resolve_class_ids(detector, KEEP_CLASSES)

    per_frame_features = []
    per_frame_boxes = []
    per_frame_classes = []
    metadata_rows = []

    for frame_id, record in enumerate(tqdm(frame_records, desc=f"Extracting {split} features")):
        image_bgr = cv2.imread(str(record.frame_path))
        if image_bgr is None:
            raise RuntimeError(f"Could not read frame: {record.frame_path}")

        result = detector.predict(
            source=image_bgr,
            conf=DETECTION_CONF,
            classes=class_ids,
            imgsz=IMG_SIZE,
            verbose=False,
            device=device,
        )[0]

        if result.boxes is not None and len(result.boxes) > 0:
            boxes = result.boxes.xyxy.detach().cpu().numpy().astype(np.float32)
            labels = result.boxes.cls.detach().cpu().numpy().astype(np.int64)
        else:
            boxes = np.empty((0, 4), dtype=np.float32)
            labels = np.empty((0,), dtype=np.int64)

        if boxes.shape[0] > 0:
            widths = boxes[:, 2] - boxes[:, 0]
            heights = boxes[:, 3] - boxes[:, 1]
            keep_mask = (widths * heights) >= MIN_BOX_AREA
            boxes = boxes[keep_mask]
            labels = labels[keep_mask]

        crops = []
        if boxes.shape[0] > 0:
            for box in boxes:
                crops.append(crop_box(image_bgr, box, PADDING_RATIO))
        elif USE_FULL_FRAME_FALLBACK:
            height, width = image_bgr.shape[:2]
            boxes = np.asarray([[0, 0, width, height]], dtype=np.float32)
            labels = np.asarray([-1], dtype=np.int64)
            crops.append(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))

        if crops:
            features = clip_encode(crops, clip_model, preprocess, device, CLIP_BATCH_SIZE)
        else:
            features = np.empty((clip_dim, 0), dtype=np.float32).reshape(0, clip_dim)

        per_frame_features.append(features)
        per_frame_boxes.append(boxes.astype(np.float32))
        per_frame_classes.append(labels.astype(np.int64))
        metadata_rows.append((frame_id, record.video_name, record.frame_index, str(record.frame_path), int(features.shape[0])))

    np.save(features_path, np.asarray(per_frame_features, dtype=object))
    np.save(boxes_path, np.asarray(per_frame_boxes, dtype=object))
    np.save(classes_path, np.asarray(per_frame_classes, dtype=object))
    save_metadata(metadata_path, metadata_rows)
    print(f"Saved {split} features to {split_output_dir}")


def load_feature_bank(features_path: Path):
    feature_list = np.load(features_path, allow_pickle=True)
    return [np.asarray(item, dtype=np.float32) for item in feature_list]


def flatten_feature_bank(feature_list):
    non_empty = [item for item in feature_list if item.size > 0]
    if not non_empty:
        raise RuntimeError("No features were found in the requested split.")
    return np.concatenate(non_empty, axis=0).astype(np.float32)


def maybe_subsample(features: np.ndarray, max_features: int, seed: int = 7):
    if max_features <= 0 or features.shape[0] <= max_features:
        return features
    rng = np.random.default_rng(seed)
    indices = rng.choice(features.shape[0], size=max_features, replace=False)
    return features[indices]


def load_test_ground_truth(dataset_root: Path):
    masks_root = dataset_root / "testing" / "test_frame_mask"
    if not masks_root.exists():
        raise FileNotFoundError(f"Ground-truth directory not found: {masks_root}")

    all_labels = []
    clip_lengths = []
    total = 0
    for mask_path in sorted(masks_root.glob("*.npy"), key=lambda path: path.name):
        labels = np.load(mask_path).astype(np.int64).reshape(-1)
        all_labels.append(labels)
        total += labels.shape[0]
        clip_lengths.append(total)

    if not all_labels:
        raise RuntimeError(f"No frame-mask files found in: {masks_root}")

    return np.concatenate(all_labels, axis=0), np.asarray(clip_lengths, dtype=np.int64)


def macro_auc(scores, labels, clip_lengths):
    scores = np.asarray(scores)
    labels = np.asarray(labels)
    prev = 0
    aucs = []
    for end in clip_lengths:
        video_scores = scores[prev:end]
        video_labels = labels[prev:end]
        padded_labels = np.concatenate(([0], video_labels, [1]))
        padded_scores = np.concatenate(([0.0], video_scores, [sys.float_info.max]))
        aucs.append(float(roc_auc_score(padded_labels, padded_scores)))
        prev = end
    return float(np.mean(aucs))


def score_test_frames(neighbors: int = K_NEIGHBORS, max_train_features: int = MAX_TRAIN_FEATURES):
    train_features = load_feature_bank(OUTPUT_ROOT / "train" / "features.npy")
    test_features = load_feature_bank(OUTPUT_ROOT / "test" / "features.npy")

    train_matrix = flatten_feature_bank(train_features)
    train_matrix = maybe_subsample(train_matrix, max_train_features)
    index = faiss.IndexFlatIP(train_matrix.shape[1])
    index.add(train_matrix.astype(np.float32))

    frame_scores = []
    k = min(neighbors, index.ntotal)
    if k < 1:
        raise RuntimeError("The kNN index is empty.")

    for frame_features in tqdm(test_features, desc="Scoring test frames"):
        if frame_features.size == 0:
            frame_scores.append(0.0)
            continue
        similarities, _ = index.search(frame_features.astype(np.float32), k)
        object_scores = 1.0 - similarities.mean(axis=1)
        frame_scores.append(float(object_scores.max()))

    frame_scores = np.asarray(frame_scores, dtype=np.float32)
    np.save(OUTPUT_ROOT / "test_frame_scores.npy", frame_scores)

    results = {
        "neighbors": int(neighbors),
        "train_features_indexed": int(index.ntotal),
        "test_frames_scored": int(frame_scores.shape[0]),
    }

    try:
        labels, clip_lengths = load_test_ground_truth(DATA_ROOT)
        if labels.shape[0] != frame_scores.shape[0]:
            raise RuntimeError(f"Label/frame mismatch: loaded {labels.shape[0]} labels but scored {frame_scores.shape[0]} frames.")
        results["micro_auc"] = float(roc_auc_score(labels, frame_scores))
        results["macro_auc"] = float(macro_auc(frame_scores, labels, clip_lengths))
    except FileNotFoundError:
        results["note"] = "Ground-truth test_frame_mask files were not found, so AUC was skipped."

    with (OUTPUT_ROOT / "results.json").open("w", encoding="utf-8") as handle:
        json.dump(results, handle, indent=2)

    return results


device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, _, preprocess = open_clip.create_model_and_transforms(CLIP_MODEL, pretrained=CLIP_PRETRAINED)
clip_model = clip_model.to(device)
clip_model.eval()
clip_dim = int(getattr(clip_model.visual, "output_dim", 512))
detector = YOLO(DETECTOR_WEIGHTS)

print("Using device:", device)
print("CLIP output dimension:", clip_dim)


In [ ]:
extract_split_features("train")
extract_split_features("test")


In [ ]:
results = score_test_frames()
results


In [ ]:
import matplotlib.pyplot as plt

frame_scores = np.load(OUTPUT_ROOT / "test_frame_scores.npy")
plt.figure(figsize=(16, 4))
plt.plot(frame_scores)
plt.title("Test frame anomaly scores")
plt.xlabel("Frame index")
plt.ylabel("Anomaly score")
plt.show()


## Where the outputs go

- Features: `/content/artifacts/clip_knn_baseline/shanghaitech/train/features.npy`
- Test features: `/content/artifacts/clip_knn_baseline/shanghaitech/test/features.npy`
- Per-frame scores: `/content/artifacts/clip_knn_baseline/shanghaitech/test_frame_scores.npy`
- Metrics: `/content/artifacts/clip_knn_baseline/shanghaitech/results.json`

For quick smoke tests before a full run, set `VIDEO_LIMIT` or `FRAME_LIMIT` near the top of the notebook.
